# Notebook 03 — Revenue Test + Bootstrap CI
**Skenario:** Apakah new page meningkatkan revenue per user?  
**Tests:** Two-sample T-test + Bootstrap Confidence Interval (10,000 iterasi)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import json
from scipy.stats import ttest_ind
from statsmodels.stats.proportion import proportions_ztest, proportion_confint
from scipy.stats import norm
from pathlib import Path

OUT     = Path('../output')
FIGURES = OUT / 'figures'

BLUE  = '#2563EB'
RED   = '#DC2626'
GREEN = '#059669'
LIGHT = '#93C5FD'
GRAY  = '#6B7280'
ALPHA = 0.05
N_BOOT = 10_000

df    = pd.read_parquet(OUT / 'df_clean.parquet')
ctrl  = df[df['group'] == 'control']['revenue'].values
treat = df[df['group'] == 'treatment']['revenue'].values

print(f'Control   n={len(ctrl):,}  mean={ctrl.mean():.4f}  median={np.median(ctrl):.4f}')
print(f'Treatment n={len(treat):,}  mean={treat.mean():.4f}  median={np.median(treat):.4f}')

## A. Statistik Deskriptif Revenue

In [ ]:
rev_stats = df.groupby('group')['revenue'].agg(['mean','median','std','min','max']).round(3)
print('Revenue statistics per grup:')
print(rev_stats)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# KDE overlay
# Plot distribusi hanya untuk revenue > 0 (converter) agar lebih informatif
ctrl_conv  = df[(df['group']=='control')   & (df['converted']==1)]['revenue']
treat_conv = df[(df['group']=='treatment') & (df['converted']==1)]['revenue']

ctrl_conv.plot.kde(ax=axes[0],  color=BLUE, linewidth=2, label=f'Control (n={len(ctrl_conv):,})')
treat_conv.plot.kde(ax=axes[0], color=RED,  linewidth=2, label=f'Treatment (n={len(treat_conv):,})')
axes[0].set_xlabel('Revenue per Converter (USD)')
axes[0].set_title('Distribusi Revenue — Converter Only\n(non-converter = $0, tidak ditampilkan)', fontweight='bold')
axes[0].legend()
axes[0].set_xlim(0)

# Box plot revenue per user (semua, termasuk $0)
df.boxplot(column='revenue', by='group', ax=axes[1],
           boxprops=dict(color=BLUE),
           medianprops=dict(color=RED, linewidth=2))
axes[1].set_title('Revenue per User (semua pengguna)', fontweight='bold')
axes[1].set_xlabel('Grup')
axes[1].set_ylabel('Revenue (USD)')
plt.suptitle('')

plt.tight_layout()
plt.savefig(FIGURES / 'C_revenue_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: C_revenue_distribution.png')

## B. Two-sample T-test

In [ ]:
t_stat, p_value = ttest_ind(treat, ctrl, equal_var=False)  # Welch's t-test

# Cohen's d (effect size)
pooled_std = np.sqrt((ctrl.std()**2 + treat.std()**2) / 2)
cohens_d   = (treat.mean() - ctrl.mean()) / pooled_std

print('=== TWO-SAMPLE T-TEST (Welch) ===')
print(f't-statistic : {t_stat:.4f}')
print(f'p-value     : {p_value:.4f}')
print(f"Cohen's d   : {cohens_d:.4f}")
print(f"  → Effect size: {'Kecil (<0.2)' if abs(cohens_d) < 0.2 else 'Sedang (0.2-0.8)' if abs(cohens_d) < 0.8 else 'Besar (>0.8)'}")
print(f'\nMean diff   : ${treat.mean() - ctrl.mean():+.4f} per user')

if p_value < ALPHA:
    print(f'\n✅ TOLAK H₀ — perbedaan revenue signifikan (p={p_value:.4f})')
else:
    print(f'\n❌ GAGAL TOLAK H₀ — perbedaan revenue TIDAK signifikan (p={p_value:.4f})')

## C. Bootstrap Confidence Interval (10,000 iterasi)

In [ ]:
np.random.seed(42)
boot_diffs = np.array([
    np.mean(np.random.choice(treat, len(treat), replace=True)) -
    np.mean(np.random.choice(ctrl,  len(ctrl),  replace=True))
    for _ in range(N_BOOT)
])

boot_ci = np.percentile(boot_diffs, [2.5, 97.5])
contains_zero = boot_ci[0] < 0 < boot_ci[1]

print(f'=== BOOTSTRAP CI (n={N_BOOT:,} iterasi) ===')
print(f'Mean diff observed : ${boot_diffs.mean():+.4f}')
print(f'95% CI             : [${boot_ci[0]:+.4f}, ${boot_ci[1]:+.4f}]')
print(f'CI mengandung 0?   : {contains_zero}')
print(f'\n→ {"Perbedaan TIDAK signifikan (CI mengandung 0)" if contains_zero else "Perbedaan SIGNIFIKAN (CI tidak mengandung 0)"}')

fig, ax = plt.subplots(figsize=(12, 5))

ax.hist(boot_diffs, bins=80, color=LIGHT, edgecolor='white', linewidth=0.3)
ax.axvline(0, color=GRAY, linestyle='--', linewidth=2, label='H₀: diff=0')
ax.axvline(boot_ci[0], color=RED, linestyle='--', linewidth=1.5)
ax.axvline(boot_ci[1], color=RED, linestyle='--', linewidth=1.5)
ax.axvspan(boot_ci[0], boot_ci[1], alpha=0.15, color=RED, label=f'95% CI: [${boot_ci[0]:+.4f}, ${boot_ci[1]:+.4f}]')
ax.axvline(boot_diffs.mean(), color=BLUE, linewidth=2.5, label=f'Mean diff: ${boot_diffs.mean():+.4f}')

ax.set_xlabel('Mean Revenue Difference (Treatment − Control, USD)')
ax.set_ylabel('Frekuensi')
ax.set_title(f'Bootstrap Distribution of Mean Difference ({N_BOOT:,} iterasi)', fontweight='bold')
ax.legend(fontsize=10)

plt.tight_layout()
plt.savefig(FIGURES / 'D_bootstrap_ci.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: D_bootstrap_ci.png')

## D. Business Decision & Revenue Projection

In [ ]:
daily_traffic    = df.groupby(df['timestamp'].dt.date)['user_id'].count().mean()
mean_diff        = treat.mean() - ctrl.mean()
daily_rev_uplift = mean_diff * daily_traffic
annual_uplift    = daily_rev_uplift * 365

verdict_rev = 'GO ✅' if p_value < ALPHA else 'NO-GO ❌'

print('=== BUSINESS DECISION (Revenue) ===')
print(f'Verdict        : {verdict_rev}')
print(f'Alasan         : p-value={p_value:.4f}, Bootstrap CI [{boot_ci[0]:+.4f}, {boot_ci[1]:+.4f}]')
print(f'\nEstimasi impact:')
print(f'  Revenue uplift/user/hari : ${mean_diff:+.4f}')
print(f'  Daily traffic            : {daily_traffic:,.0f} users/hari')
print(f'  Revenue uplift/hari      : ${daily_rev_uplift:+,.0f}')
print(f'  Revenue uplift/tahun     : ${annual_uplift:+,.0f}')

# Simpan semua hasil ke JSON
results = {
    'conversion_test': {
        'p_ctrl':   round(float(df[df['group']=='control']['converted'].mean()), 4),
        'p_treat':  round(float(df[df['group']=='treatment']['converted'].mean()), 4),
    },
    'revenue_test': {
        't_stat':   round(float(t_stat), 4),
        'p_value':  round(float(p_value), 4),
        'cohens_d': round(float(cohens_d), 4),
        'bootstrap_ci_lower': round(float(boot_ci[0]), 4),
        'bootstrap_ci_upper': round(float(boot_ci[1]), 4),
        'mean_diff': round(float(mean_diff), 4),
        'annual_uplift_usd': round(float(annual_uplift), 2),
        'verdict': verdict_rev
    }
}

with open(OUT / 'test_results.json', 'w') as f:
    json.dump(results, f, indent=2)
print('\nSaved: test_results.json')